In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import re
import datetime
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
import pickle

In [2]:
CSV_PATH = "../csv/"
K_COLS = ['着', 'RaceID', '選手登番', '展示', 'スタートタイミング']

In [3]:
k_files = glob.glob("../csv/K*")
b_files = glob.glob("../csv/B*")

In [4]:
def concat_files(files):
    tmp = pd.DataFrame()
    for file in tqdm(files):
        df = pd.read_csv(file, index_col=0)
        tmp = pd.concat([tmp,df])
    return tmp

In [5]:
kdf = concat_files(k_files)
bdf = concat_files(b_files)

100%|██████████| 791/791 [00:50<00:00, 15.60it/s]


In [128]:
df = pd.merge(bdf,kdf.loc[:,K_COLS],on = ['RaceID','選手登番'])

In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 671786 entries, 0 to 671785
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   艇番         671786 non-null  int64  
 1   選手登番       671786 non-null  int64  
 2   選手名        671786 non-null  object 
 3   年齢         671786 non-null  int64  
 4   支部         671786 non-null  object 
 5   体重         671786 non-null  int64  
 6   級別         671786 non-null  object 
 7   全国勝率       671786 non-null  float64
 8   全国2連率      671786 non-null  float64
 9   当地勝率       671786 non-null  float64
 10  当地2連率      671786 non-null  float64
 11  モーター2連率    671786 non-null  float64
 12  ボート2連率     671786 non-null  float64
 13  RaceID     671786 non-null  object 
 14  場所         671786 non-null  int64  
 15  R          671786 non-null  object 
 16  着          671786 non-null  int64  
 17  展示         671786 non-null  float64
 18  スタートタイミング  671786 non-null  float64
dtypes: float64(8), int64(6)

In [130]:
def LabelEncoding(df,col):
    #インスタンス
    LE = LabelEncoder()

    #Label エンコーディング
    LE.fit_transform(df[col].values)

    pickle.dump(LE, open('../model/' + col + '_LE.pickle', 'wb'))

    #データ変換
    le = LE.fit_transform(df[col].values)
    return le

def zero_padding(txt):
    l = re.findall(r"\d+", txt)
    l = [int(s) for s in l]
    date = datetime.date(*l[:3])
    raceid = str(date) + '-' + str(l[3]).zfill(2) + '-' + str(l[4]).zfill(2)
    return raceid


In [131]:
df_encoded = df
df_encoded['支部'] = LabelEncoding(df,'支部')
df_encoded['級別'] = LabelEncoding(df, '級別')
df_encoded['R'] = df['R'].replace('R', '', regex=True).astype('int')
df_encoded['RaceID'] = df_encoded['RaceID'].replace('_','/',regex=True).replace('R','',regex=True)

In [138]:
zero = []
for n,i in enumerate(tqdm(df_encoded['RaceID'])):
    zero.append(zero_padding(i))
df_encoded['RaceID'] = zero

100%|██████████| 671786/671786 [00:05<00:00, 121729.14it/s]


In [144]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 671786 entries, 0 to 671785
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   艇番         671786 non-null  int64  
 1   選手登番       671786 non-null  int64  
 2   選手名        671786 non-null  object 
 3   年齢         671786 non-null  int64  
 4   支部         671786 non-null  int64  
 5   体重         671786 non-null  int64  
 6   級別         671786 non-null  int64  
 7   全国勝率       671786 non-null  float64
 8   全国2連率      671786 non-null  float64
 9   当地勝率       671786 non-null  float64
 10  当地2連率      671786 non-null  float64
 11  モーター2連率    671786 non-null  float64
 12  ボート2連率     671786 non-null  float64
 13  RaceID     671786 non-null  object 
 14  場所         671786 non-null  int64  
 15  R          671786 non-null  int64  
 16  着          671786 non-null  int64  
 17  展示         671786 non-null  float64
 18  スタートタイミング  671786 non-null  float64
dtypes: float64(8), int64(9)

In [143]:
os.makedirs('../train_csv',exist_ok=True)
df_encoded.to_csv('../train_csv/train.csv')